# ENHANCED DATA PREPROCESSING & LSTM TRAINING
## Purpose: Clean data, prevent overfitting/underfitting

**IMPORTANT: Column Name Mapping**

The actual data has different column names. This notebook maps them:

| Standard | Your File | Use |
|----------|-----------|-----|
| `admid` | `adm_id` | Admin unit ID |
| `year_month` | `date` or `time` | Date column |
| `vimavg` | `vim_avg` | NDVI vegetation index |
| `SPI3` | `SPI_3` | 3-month SPI |
| `SPI6` | `SPI_6` | 6-month SPI |

---

This notebook provides a **production-ready pipeline** for:
1. **Data Cleaning**: Remove duplicates, handle missing values, detect outliers
2. **Data Validation**: Check value ranges, temporal continuity, multicollinearity
3. **Feature Engineering**: Create lagged features, seasonal decomposition
4. **LSTM Training**: Prevent overfitting/underfitting with regularization
5. **Model Evaluation**: Comprehensive diagnostics and visualization
6. **Architecture Comparison**: Test multiple LSTM configurations
7. **Cross-Validation**: Walk-forward validation for time series
8. **Model Interpretation**: Residual analysis and feature importance

## Step 0: Import Libraries

In [ ]:
!pip install tensorflow



  Using cached tensorflow-2.20.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.13.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metada

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import median_abs_deviation, norm, probplot
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

---

# STEP 1: COLUMN NAME MAPPING

**Map your actual column names to standard names**

In [3]:
# ============================================
# COLUMN NAME MAPPING
# ============================================

# actual file names
NDVI_FILE = 'eth-ndvi-subnat-full.csv'
SPI_FILE = 'ethiopia_rainfall_spi.csv'

# Column name mappings
NDVI_MAPPING = {
    'adm_id': 'admid',
    'date': 'year_month',
    'vim_avg': 'vimavg'
}

SPI_MAPPING = {
    'time': 'year_month',
    'SPI_3': 'SPI3',
    'SPI_6': 'SPI6'
}

NDVI_KEEP_COLS = ['admid', 'year_month', 'vimavg', 'precipitation']

print("✓ Column mapping configured")
print(f"  NDVI mapping: {NDVI_MAPPING}")
print(f"  SPI mapping: {SPI_MAPPING}")

✓ Column mapping configured
  NDVI mapping: {'adm_id': 'admid', 'date': 'year_month', 'vim_avg': 'vimavg'}
  SPI mapping: {'time': 'year_month', 'SPI_3': 'SPI3', 'SPI_6': 'SPI6'}


---

# PART 1: ENHANCED DATA CLEANING

In [4]:
def load_and_rename_data(ndvi_file, spi_file, ndvi_mapping, spi_mapping):
    print("="*60)
    print("STEP 1: LOADING AND RENAMING DATA")
    print("="*60)
    
    ndvi = pd.read_csv(ndvi_file)
    spi = pd.read_csv(spi_file)
    
    print(f"\n🔄 Original NDVI columns: {ndvi.columns.tolist()}")
    print(f"🔄 Original SPI columns: {spi.columns.tolist()}")
    
    ndvi = ndvi.rename(columns=ndvi_mapping)
    spi = spi.rename(columns=spi_mapping)
    
    print(f"\n✓ Renamed NDVI columns: {ndvi.columns.tolist()}")
    print(f"✓ Renamed SPI columns: {spi.columns.tolist()}")
    
    ndvi['year_month'] = pd.to_datetime(ndvi['year_month'])
    spi['year_month'] = pd.to_datetime(spi['year_month'])
    
    ndvi['year_month'] = ndvi['year_month'].dt.to_period('M')
    spi['year_month'] = spi['year_month'].dt.to_period('M')
    
    print(f"\nNDVI shape: {ndvi.shape}")
    print(f"SPI shape: {spi.shape}")
    print(f"\nNDVI date range: {ndvi['year_month'].min()} to {ndvi['year_month'].max()}")
    print(f"SPI date range: {spi['year_month'].min()} to {spi['year_month'].max()}")
    print(f"\nNDVI missing values:\n{ndvi.isnull().sum()}")
    print(f"\nSPI missing values:\n{spi.isnull().sum()}")
    
    return ndvi, spi

In [5]:
def remove_duplicates(df, subset_cols):
    print("\n" + "="*60)
    print("STEP 2: REMOVING DUPLICATES")
    print("="*60)
    
    initial_rows = len(df)
    df = df.drop_duplicates(subset=subset_cols, keep='first')
    removed = initial_rows - len(df)
    
    print(f"Rows removed: {removed}")
    print(f"Remaining rows: {len(df)}")
    
    return df

In [6]:
def validate_value_ranges(df):
    print("\n" + "="*60)
    print("STEP 3: VALIDATING VALUE RANGES")
    print("="*60)
    
    if 'vimavg' in df.columns:
        ndvi_range = df['vimavg'].describe()
        print(f"\nNDVI Statistics:")
        print(ndvi_range)
        
        below_zero = (df['vimavg'] < 0).sum()
        above_one = (df['vimavg'] > 1).sum()
        
        if below_zero > 0:
            print(f"⚠️  WARNING: {below_zero} NDVI values below 0")
        if above_one > 0:
            print(f"⚠️  WARNING: {above_one} NDVI values above 1")
    
    if 'SPI3' in df.columns:
        spi_range = df['SPI3'].describe()
        print(f"\nSPI3 Statistics:")
        print(spi_range)
        
        extreme_spi = ((df['SPI3'] < -4) | (df['SPI3'] > 4)).sum()
        if extreme_spi > 0:
            print(f"⚠️  WARNING: {extreme_spi} extreme SPI values (outside -4 to +4)")
    
    return df

In [7]:
def detect_and_handle_outliers(df, numeric_cols, method='modified_zscore', threshold=3.5):
    print("\n" + "="*60)
    print(f"STEP 4: OUTLIER DETECTION ({method.upper()})")
    print("="*60)
    
    outliers_mask = pd.DataFrame(False, index=df.index, columns=numeric_cols)
    
    if method == 'modified_zscore':
        for col in numeric_cols:
            median = df[col].median()
            mad = median_abs_deviation(df[col], nan_policy='propagate')
            
            if mad == 0:
                print(f"\n⚠️  {col}: MAD is zero (constant values), using IQR instead")
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                col_outliers = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)
            else:
                modified_z_scores = 0.6745 * (df[col] - median) / mad
                col_outliers = np.abs(modified_z_scores) > threshold
            
            outliers_mask[col] = col_outliers
            
            n_outliers = col_outliers.sum()
            if n_outliers > 0:
                print(f"\n{col}: {n_outliers} outliers detected ({n_outliers/len(df)*100:.2f}%)")
                print(f"  Median: {median:.3f}, MAD: {mad:.3f}")
                print(f"  Actual range: [{df[col].min():.3f}, {df[col].max():.3f}]")
    
    print(f"\nAction: Flagging outliers (not removing - will use robust training)")
    df['is_outlier'] = outliers_mask.any(axis=1)
    
    return df

In [8]:
def handle_missing_values_improved(df, numeric_cols, strategy='interpolate'):
    print("\n" + "="*60)
    print(f"STEP 5: HANDLING MISSING VALUES ({strategy.upper()})")
    print("="*60)
    
    print(f"\nMissing values before:")
    print(df[numeric_cols].isnull().sum())
    
    if strategy == 'drop':
        df = df.dropna(subset=numeric_cols)
    
    elif strategy == 'forward_fill':
        if 'admid' in df.columns:
            for col in numeric_cols:
                df[col] = df.groupby('admid')[col].transform(lambda x: x.fillna(method='ffill'))
        else:
            df[numeric_cols] = df[numeric_cols].fillna(method='ffill')
    
    elif strategy == 'interpolate':
        if 'admid' in df.columns:
            for col in numeric_cols:
                df[col] = df.groupby('admid')[col].transform(
                    lambda x: x.interpolate(method='linear', limit_direction='both', limit=3)
                )
        else:
            df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')
    
    elif strategy == 'mean':
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    
    df = df.dropna(subset=numeric_cols)
    
    print(f"\nMissing values after:")
    print(df[numeric_cols].isnull().sum())
    print(f"Rows remaining: {len(df)}")
    
    return df

In [23]:
def check_temporal_continuity(df):
    """
    Verify no large gaps in time series (FIXED for Period data)
    """
    print("\n" + "="*60)
    print("STEP 6: CHECKING TEMPORAL CONTINUITY")
    print("="*60)
    
    if 'year_month' not in df.columns:
        print("⚠️  'year_month' column not found - skipping temporal check")
        return df
    
    # Convert Period to integer for diff calculation
    if 'admid' in df.columns:
        for admid in df['admid'].unique()[:5]:  # Check first 5 admin units
            admin_data = df[df['admid'] == admid].sort_values('year_month')
            if len(admin_data) > 1:
                # ✅ FIXED: Convert Period to int, then check for gaps != 1 month
                period_nums = admin_data['year_month'].astype(int)
                gap_count = (period_nums.diff() != 1).sum() - 1  # !=1 means gap >1 month
                if gap_count > 0:
                    print(f"⚠️  Admin {admid}: {gap_count} gaps detected")
    
    print("✓ Temporal continuity check complete")
    return df


In [10]:
def check_multicollinearity(df, numeric_cols):
    print("\n" + "="*60)
    print("STEP 7: MULTICOLLINEARITY CHECK (VIF)")
    print("="*60)
    
    df_clean = df[numeric_cols].dropna()
    
    if len(df_clean) == 0:
        print("⚠️  No valid data for VIF calculation")
        return None
    
    print(f"\nVariance Inflation Factor (VIF):")
    print("VIF > 10: High multicollinearity")
    print("VIF > 5: Moderate multicollinearity")
    print("VIF < 5: Acceptable\n")
    
    vif_data = pd.DataFrame()
    vif_data["Feature"] = numeric_cols
    vif_data["VIF"] = [variance_inflation_factor(df_clean.values, i) 
                       for i in range(df_clean.shape[1])]
    
    print(vif_data.to_string(index=False))
    
    high_vif = vif_data[vif_data['VIF'] > 5]
    if len(high_vif) > 0:
        print(f"\n⚠️  High multicollinearity columns:")
        print(high_vif['Feature'].tolist())
    
    return vif_data

In [11]:
def merge_and_clean_data(ndvi, spi):
    print("\n" + "="*60)
    print("STEP 8: MERGING AND FEATURE ENGINEERING")
    print("="*60)
    
    merged = ndvi.merge(spi, on='year_month', how='inner')
    print(f"\nMerged shape: {merged.shape}")
    
    merged = merged.sort_values(['admid', 'year_month']).reset_index(drop=True)
    
    numeric_features = ['SPI3', 'SPI6', 'precipitation']
    for lag in [1, 2, 3]:
        for feat in numeric_features:
            if feat in merged.columns:
                merged[f'{feat}_lag{lag}'] = merged.groupby('admid')[feat].shift(lag)
    
    print(f"Features after lagging: {merged.shape[1]} columns")
    
    return merged

In [12]:
def seasonal_decomposition_check(df, target_col='vimavg', period=12):
    print("\n" + "="*60)
    print("STEP 9: SEASONAL DECOMPOSITION")
    print("="*60)
    
    if target_col not in df.columns:
        print(f"⚠️  {target_col} not found")
        return
    
    ts_data = df.groupby('year_month')[target_col].mean()
    
    if len(ts_data) < 2 * period:
        print(f"⚠️  Not enough data for decomposition (need {2*period}, have {len(ts_data)})")
        return
    
    try:
        decomposition = seasonal_decompose(ts_data, model='additive', period=period)
        
        fig, axes = plt.subplots(4, 1, figsize=(12, 10))
        
        decomposition.observed.plot(ax=axes[0], title='Observed')
        axes[0].set_ylabel('NDVI')
        
        decomposition.trend.plot(ax=axes[1], title='Trend')
        axes[1].set_ylabel('Trend')
        
        decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
        axes[2].set_ylabel('Seasonal')
        
        decomposition.resid.plot(ax=axes[3], title='Residual')
        axes[3].set_ylabel('Residual')
        
        plt.tight_layout()
        plt.savefig('seasonal_decomposition.png', dpi=300, bbox_inches='tight')
        print("\n✓ Seasonal decomposition plot saved: seasonal_decomposition.png")
        plt.close()
        
    except Exception as e:
        print(f"⚠️  Could not perform decomposition: {str(e)}")

---

# PART 2: OVERFITTING/UNDERFITTING PREVENTION

In [26]:
def create_sequences(X, y, lookback=6):
    Xs, ys = [], []
    for i in range(len(X) - lookback):
        Xs.append(X[i:i+lookback])
        ys.append(y[i+lookback])
    return np.array(Xs), np.array(ys)

def stratified_temporal_split(X_seq, y_seq, train_ratio=0.6, val_ratio=0.2):
    print("\n" + "="*60)
    print("STEP 10: TEMPORAL TRAIN/VAL/TEST SPLIT")
    print("="*60)
    
    n_samples = len(X_seq)
    train_end = int(n_samples * train_ratio)
    val_end = train_end + int(n_samples * val_ratio)
    
    X_train = X_seq[:train_end]
    y_train = y_seq[:train_end]
    X_val = X_seq[train_end:val_end]
    y_val = y_seq[train_end:val_end]
    X_test = X_seq[val_end:]
    y_test = y_seq[val_end:]
    
    print(f"\nTrain: {len(X_train)} samples ({len(X_train)/n_samples*100:.1f}%)")
    print(f"Val:   {len(X_val)} samples ({len(X_val)/n_samples*100:.1f}%)")
    print(f"Test:  {len(X_test)} samples ({len(X_test)/n_samples*100:.1f}%)")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [14]:
def build_lstm_model_advanced(input_shape, architecture='balanced'):
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
    from tensorflow.keras.regularizers import l2
    from tensorflow.keras.optimizers import Adam
    
    print("\n" + "="*60)
    print(f"STEP 11: BUILDING LSTM MODEL ({architecture.upper()})")
    print("="*60)
    
    model = Sequential()
    
    if architecture == 'light':
        model.add(LSTM(32, input_shape=input_shape, return_sequences=True))
        model.add(BatchNormalization())
        model.add(Dropout(0.2))
        model.add(LSTM(16))
        model.add(Dropout(0.2))
        model.add(Dense(1))
        print("Light architecture: 32→16 (for small datasets)")
    
    elif architecture == 'balanced':
        model.add(LSTM(64, input_shape=input_shape, return_sequences=True))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(LSTM(32, return_sequences=True))
        model.add(BatchNormalization())
        model.add(Dropout(0.3))
        model.add(LSTM(16))
        model.add(Dropout(0.2))
        model.add(Dense(16, activation='relu', kernel_regularizer=l2(0.001)))
        model.add(Dropout(0.2))
        model.add(Dense(1))
        print("Balanced architecture: 64→32→16 with L2 regularization")
    
    elif architecture == 'heavy':
        model.add(LSTM(128, input_shape=input_shape, return_sequences=True))
        model.add(BatchNormalization())
        model.add(Dropout(0.4))
        model.add(LSTM(64, return_sequences=True))
        model.add(BatchNormalization())
        model.add(Dropout(0.4))
        model.add(LSTM(32))
        model.add(Dropout(0.3))
        model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.001)))
        model.add(Dropout(0.3))
        model.add(Dense(1))
        print("Heavy architecture: 128→64→32 with strong regularization")
    
    optimizer = Adam(learning_rate=0.001, clipvalue=1.0, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    print(f"\nModel parameters: {model.count_params():,}")
    print(f"Gradient clipping: clipvalue=1.0, clipnorm=1.0")
    
    return model

In [15]:
def train_model_with_monitoring(model, X_train, y_train, X_val, y_val, epochs=100):
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
    
    print("\n" + "="*60)
    print("STEP 12: TRAINING WITH ADVANCED REGULARIZATION")
    print("="*60)
    
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            'best_model.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

In [16]:
def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, scaler_y=None):
    print("\n" + "="*60)
    print("STEP 13: MODEL EVALUATION & DIAGNOSTICS")
    print("="*60)
    
    y_train_pred = model.predict(X_train, verbose=0)
    y_val_pred = model.predict(X_val, verbose=0)
    y_test_pred = model.predict(X_test, verbose=0)
    
    def get_metrics(y_true, y_pred):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        return rmse, mae, r2
    
    rmse_train, mae_train, r2_train = get_metrics(y_train, y_train_pred.flatten())
    rmse_val, mae_val, r2_val = get_metrics(y_val, y_val_pred.flatten())
    rmse_test, mae_test, r2_test = get_metrics(y_test, y_test_pred.flatten())
    
    print(f"\nTrain - RMSE: {rmse_train:.4f}, MAE: {mae_train:.4f}, R²: {r2_train:.4f}")
    print(f"Val   - RMSE: {rmse_val:.4f}, MAE: {mae_val:.4f}, R²: {r2_val:.4f}")
    print(f"Test  - RMSE: {rmse_test:.4f}, MAE: {mae_test:.4f}, R²: {r2_test:.4f}")
    
    train_val_gap = rmse_val - rmse_train
    print(f"\nRMSE gap (Val - Train): {train_val_gap:.4f}")
    
    if rmse_train > 0.1:
        print("⚠️  HIGH TRAINING ERROR - Model is UNDERFITTING")
        print("   Recommendation: Use lighter model or more features")
    elif train_val_gap > 0.05:
        print("⚠️  VALIDATION ERROR >> TRAINING ERROR - Model is OVERFITTING")
        print("   Recommendation: Increase dropout or reduce model complexity")
    else:
        print("✓ Good generalization - balanced model")
    
    metrics = {
        'train': {'rmse': rmse_train, 'mae': mae_train, 'r2': r2_train},
        'val': {'rmse': rmse_val, 'mae': mae_val, 'r2': r2_val},
        'test': {'rmse': rmse_test, 'mae': mae_test, 'r2': r2_test}
    }
    
    return metrics, y_train_pred, y_val_pred, y_test_pred

In [17]:
def plot_training_history(history):
    print("\n" + "="*60)
    print("STEP 14: PLOTTING TRAINING HISTORY")
    print("="*60)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].set_title('Training History - Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('Training History - MAE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    print("✓ Training history plot saved: training_history.png")
    plt.close()

In [18]:
def plot_predictions_vs_actual(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):
    print("\n" + "="*60)
    print("STEP 15: PLOTTING PREDICTIONS VS ACTUAL")
    print("="*60)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    axes[0].plot(y_train, label='Actual', alpha=0.7, linewidth=1.5)
    axes[0].plot(y_train_pred.flatten(), label='Predicted', alpha=0.7, linewidth=1.5)
    axes[0].set_title('Train Set - Actual vs Predicted')
    axes[0].set_ylabel('NDVI (normalized)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(y_val, label='Actual', alpha=0.7, linewidth=1.5)
    axes[1].plot(y_val_pred.flatten(), label='Predicted', alpha=0.7, linewidth=1.5)
    axes[1].set_title('Validation Set - Actual vs Predicted')
    axes[1].set_ylabel('NDVI (normalized)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(y_test, label='Actual', alpha=0.7, linewidth=1.5)
    axes[2].plot(y_test_pred.flatten(), label='Predicted', alpha=0.7, linewidth=1.5)
    axes[2].set_title('Test Set - Actual vs Predicted')
    axes[2].set_ylabel('NDVI (normalized)')
    axes[2].set_xlabel('Time Step')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('predictions_vs_actual.png', dpi=300, bbox_inches='tight')
    print("✓ Predictions plot saved: predictions_vs_actual.png")
    plt.close()

In [19]:
def analyze_residuals(y_true, y_pred, set_name='Test'):
    print(f"\nRESIDUAL ANALYSIS ({set_name} Set):")
    
    residuals = y_true - y_pred.flatten()
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    axes[0, 0].plot(residuals, alpha=0.7)
    axes[0, 0].axhline(y=0, color='r', linestyle='--')
    axes[0, 0].set_title(f'{set_name} Residuals Over Time')
    axes[0, 0].set_ylabel('Residual')
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
    axes[0, 1].set_title(f'{set_name} Residuals Distribution')
    axes[0, 1].set_xlabel('Residual')
    axes[0, 1].set_ylabel('Frequency')
    
    probplot(residuals, dist="norm", plot=axes[1, 0])
    axes[1, 0].set_title(f'{set_name} Q-Q Plot')
    
    plot_acf(residuals, lags=20, ax=axes[1, 1])
    axes[1, 1].set_title(f'{set_name} Autocorrelation of Residuals')
    
    plt.tight_layout()
    plt.savefig(f'residuals_{set_name.lower()}.png', dpi=300, bbox_inches='tight')
    print(f"✓ Residuals plot saved: residuals_{set_name.lower()}.png")
    plt.close()
    
    print(f"  Mean: {residuals.mean():.4f}")
    print(f"  Std: {residuals.std():.4f}")
    print(f"  Min: {residuals.min():.4f}")
    print(f"  Max: {residuals.max():.4f}")

In [21]:
print("NDVI columns:", ndvi.columns.tolist())

NDVI columns: ['year_month', 'adm_level', 'admid', 'PCODE', 'n_pixels', 'vim', 'vimavg', 'viq']


---

# MAIN EXECUTION: Run Full Pipeline

In [24]:
# ===== DATA CLEANING PHASE =====
ndvi, spi = load_and_rename_data(NDVI_FILE, SPI_FILE, NDVI_MAPPING, SPI_MAPPING)
ndvi = remove_duplicates(ndvi, ['admid', 'year_month'])
spi = remove_duplicates(spi, ['year_month'])
ndvi = validate_value_ranges(ndvi)

numeric_cols_ndvi = ['vimavg']
ndvi = detect_and_handle_outliers(ndvi, numeric_cols_ndvi, method='modified_zscore', threshold=3.5)

ndvi = handle_missing_values_improved(ndvi, numeric_cols_ndvi, strategy='interpolate')
spi = handle_missing_values_improved(spi, ['SPI3', 'SPI6'], strategy='forward_fill')

ndvi = check_temporal_continuity(ndvi)
merged = merge_and_clean_data(ndvi, spi)

numeric_cols_all = merged.select_dtypes(include=[np.number]).columns.tolist()
merged = handle_missing_values_improved(merged, numeric_cols_all, strategy='forward_fill')
merged = check_temporal_continuity(merged)

numeric_features = [col for col in numeric_cols_all if col not in ['is_outlier']]
vif_data = check_multicollinearity(merged, numeric_features)
seasonal_decomposition_check(merged, target_col='vimavg', period=12)

print("\n" + "="*60)
print("DATA CLEANING COMPLETE")
print("="*60)
print(f"\nFinal dataset shape: {merged.shape}")
print(f"Final missing values:\n{merged.isnull().sum()}")

STEP 1: LOADING AND RENAMING DATA

🔄 Original NDVI columns: ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'vim', 'vim_avg', 'viq']
🔄 Original SPI columns: ['time', 'band', 'spatial_ref', 'precipitation', 'SPI_3', 'SPI_6']

✓ Renamed NDVI columns: ['year_month', 'adm_level', 'admid', 'PCODE', 'n_pixels', 'vim', 'vimavg', 'viq']
✓ Renamed SPI columns: ['year_month', 'band', 'spatial_ref', 'precipitation', 'SPI3', 'SPI6']

NDVI shape: (76895, 8)
SPI shape: (529, 6)

NDVI date range: 2002-07 to 2025-12
SPI date range: 1981-01 to 2025-11

NDVI missing values:
year_month    0
adm_level     0
admid         0
PCODE         0
n_pixels      0
vim           0
vimavg        0
viq           0
dtype: int64

SPI missing values:
year_month       0
band             0
spatial_ref      0
precipitation    0
SPI3             2
SPI6             5
dtype: int64

STEP 2: REMOVING DUPLICATES
Rows removed: 51233
Remaining rows: 25662

STEP 2: REMOVING DUPLICATES
Rows removed: 0
Remaining rows: 529

STEP 3

In [27]:
# ===== MODEL TRAINING PHASE =====
feature_cols = [col for col in numeric_features if col != 'vimavg']
X = merged[feature_cols].values
y = merged['vimavg'].values

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

lookback = 6
X_seq, y_seq = create_sequences(X_scaled, y_scaled, lookback=lookback)

print(f"\nSequences shape: X={X_seq.shape}, y={y_seq.shape}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")


Sequences shape: X=(24564, 6, 19), y=(24564,)
Feature columns (19): ['adm_level', 'admid', 'n_pixels', 'vim', 'viq', 'band', 'spatial_ref', 'precipitation', 'SPI3', 'SPI6', 'SPI3_lag1', 'SPI6_lag1', 'precipitation_lag1', 'SPI3_lag2', 'SPI6_lag2', 'precipitation_lag2', 'SPI3_lag3', 'SPI6_lag3', 'precipitation_lag3']


In [28]:
X_train, X_val, X_test, y_train, y_val, y_test = stratified_temporal_split(
    X_seq, y_seq, train_ratio=0.6, val_ratio=0.2
)


STEP 10: TEMPORAL TRAIN/VAL/TEST SPLIT

Train: 14738 samples (60.0%)
Val:   4912 samples (20.0%)
Test:  4914 samples (20.0%)


In [ ]:
!pip install tensorflow

  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/620.8 MB 119.2 kB/s eta 1:26:15

In [29]:
input_shape = (lookback, len(feature_cols))
model = build_lstm_model_advanced(input_shape, architecture='balanced')

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
history = train_model_with_monitoring(model, X_train, y_train, X_val, y_val, epochs=150)

In [ ]:
metrics, y_train_pred, y_val_pred, y_test_pred = evaluate_model(
    model, X_train, y_train, X_val, y_val, X_test, y_test, scaler_y=scaler_y
)

In [ ]:
plot_training_history(history)
plot_predictions_vs_actual(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

In [ ]:
print("\n" + "="*60)
print("STEP 16: RESIDUAL ANALYSIS")
print("="*60)

analyze_residuals(y_train, y_train_pred, set_name='Train')
analyze_residuals(y_val, y_val_pred, set_name='Validation')
analyze_residuals(y_test, y_test_pred, set_name='Test')

In [ ]:
model.save('ndvi_spi_lstm_enhanced.h5')
print("\n✓ Model saved: ndvi_spi_lstm_enhanced.h5")

print("\n" + "="*60)
print("FULL PIPELINE COMPLETE")
print("="*60)
print("\nGenerated outputs:")
print("  - seasonal_decomposition.png")
print("  - training_history.png")
print("  - predictions_vs_actual.png")
print("  - residuals_train.png")
print("  - residuals_validation.png")
print("  - residuals_test.png")
print("  - best_model.h5 (best weights during training)")
print("  - ndvi_spi_lstm_enhanced.h5 (final model)")